<a href="https://colab.research.google.com/github/ajsarsva/video-captioning-thesis/blob/main/notebooks/day15_staratergy_A_full.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Master cell

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

if os.path.exists('/content/video-captioning-thesis'):
    %cd /content/video-captioning-thesis
    !git pull origin main
else:
    !git clone https://github.com/ajsarsva/video-captioning-thesis.git
    %cd /content/video-captioning-thesis

import sys
sys.path.append('/content/video-captioning-thesis/src')

print("✅ Ready!")

Mounted at /content/drive
Cloning into 'video-captioning-thesis'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 84 (delta 42), reused 27 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 16.83 MiB | 10.25 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/video-captioning-thesis
✅ Ready!


Check for dataset existance


In [2]:
# Check what we already have
import os

drive_videos = '/content/drive/MyDrive/thesis-data/videos/'
existing = os.listdir(drive_videos)
print(f"Videos already on Drive: {len(existing)}")

Videos already on Drive: 7010


In [3]:
# Check if previous download still exists
colab_videos = '/content/msrvtt_extracted/TrainValVideo/'

if os.path.exists(colab_videos):
    all_videos = os.listdir(colab_videos)
    print(f"Videos in Colab temp storage: {len(all_videos)}")
else:
    print("Colab temp storage was cleared — need to re-download")

Colab temp storage was cleared — need to re-download


Download dataset

In [4]:
# Restore Kaggle credentials first (already in master setup)
# Then re-download
!kaggle datasets download -d vishnutheepb/msrvtt -p /content/msrvtt_raw
!unzip -q /content/msrvtt_raw/msrvtt.zip -d /content/msrvtt_extracted
print("Done! Videos available again.")

Dataset URL: https://www.kaggle.com/datasets/vishnutheepb/msrvtt
License(s): unknown
100% 4.26G/4.26G [00:51<00:00, 88.3MB/s]

Done! Videos available again.


In [3]:
video_source = '/content/msrvtt_extracted/TrainValVideo/'
drive_videos = '/content/drive/MyDrive/thesis-data/videos/'

Copying all videos to drive

In [5]:
import shutil, os

video_source = '/content/msrvtt_extracted/TrainValVideo/'
drive_videos = '/content/drive/MyDrive/thesis-data/videos/'

all_videos = [f for f in os.listdir(video_source) if f.endswith('.mp4')]
print(f"Total videos to copy: {len(all_videos)}")

for i, vid in enumerate(all_videos):
    src = os.path.join(video_source, vid)
    dst = os.path.join(drive_videos, vid)
    shutil.copy(src, dst)
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(all_videos)} copied...")

print(f"Done! Total on Drive: {len(os.listdir(drive_videos))}")

Total videos to copy: 7010
  100/7010 copied...
  200/7010 copied...
  300/7010 copied...
  400/7010 copied...
  500/7010 copied...
  600/7010 copied...
  700/7010 copied...
  800/7010 copied...
  900/7010 copied...
  1000/7010 copied...
  1100/7010 copied...
  1200/7010 copied...
  1300/7010 copied...
  1400/7010 copied...
  1500/7010 copied...
  1600/7010 copied...
  1700/7010 copied...
  1800/7010 copied...
  1900/7010 copied...
  2000/7010 copied...
  2100/7010 copied...
  2200/7010 copied...
  2300/7010 copied...
  2400/7010 copied...
  2500/7010 copied...
  2600/7010 copied...
  2700/7010 copied...
  2800/7010 copied...
  2900/7010 copied...
  3000/7010 copied...
  3100/7010 copied...
  3200/7010 copied...
  3300/7010 copied...
  3400/7010 copied...
  3500/7010 copied...
  3600/7010 copied...
  3700/7010 copied...
  3800/7010 copied...
  3900/7010 copied...
  4000/7010 copied...
  4100/7010 copied...
  4200/7010 copied...
  4300/7010 copied...
  4400/7010 copied...
  4500/7010 co

Verification for videos in drive

In [4]:
videos_on_drive = os.listdir(drive_videos)
print(f"Total videos on Drive: {len(videos_on_drive)}")

# Check for any missing video IDs
all_ids = set(f'video{i}' for i in range(7010))
on_drive = set(v.replace('.mp4', '') for v in videos_on_drive)

missing = all_ids - on_drive
print(f"Missing videos: {len(missing)}")
if missing:
    print("Missing IDs:", sorted(missing)[:10])

Total videos on Drive: 7010
Missing videos: 0


In [5]:
import os, json

# Check video IDs we have
drive_videos = '/content/drive/MyDrive/thesis-data/videos/'
video_ids = set(v.replace('.mp4', '') for v in os.listdir(drive_videos))
print(f"Total videos on Drive: {len(video_ids)}")
print(f"Sample IDs: {sorted(list(video_ids))[:10]}")

# Check what IDs are in annotation file
with open('/content/drive/MyDrive/thesis-data/MSR_VTT.json') as f:
    data = json.load(f)

ann_ids = set(ann['image_id'] for ann in data['annotations'])
print(f"\nTotal videos in annotation file: {len(ann_ids)}")

# Find overlap
overlap = video_ids & ann_ids
print(f"Videos with annotations: {len(overlap)}")
print(f"Videos WITHOUT annotations: {len(video_ids - ann_ids)}")

Total videos on Drive: 7010
Sample IDs: ['video0', 'video1', 'video10', 'video100', 'video1000', 'video1001', 'video1002', 'video1003', 'video1004', 'video1005']

Total videos in annotation file: 10000
Videos with annotations: 7010
Videos WITHOUT annotations: 0


In [6]:
# Check the actual video ID range we have
video_ids = set(v.replace('.mp4', '') for v in os.listdir(drive_videos))

# Extract numbers
numbers = sorted([int(v.replace('video', '')) for v in video_ids])
print(f"Min video ID: video{numbers[0]}")
print(f"Max video ID: video{numbers[-1]}")
print(f"Total: {len(numbers)}")

# Check if there are any gaps
expected = set(range(numbers[0], numbers[-1]+1))
actual = set(numbers)
gaps = expected - actual
print(f"Gaps in sequence: {len(gaps)}")
if gaps:
    print(f"Missing numbers: {sorted(gaps)[:10]}")

Min video ID: video0
Max video ID: video7009
Total: 7010
Gaps in sequence: 0


Load BLIP Model

In [7]:
import json, time, os, sys
sys.path.append('/content/video-captioning-thesis/src')

from frame_extractor import extract_frames
from strategy_a_uniform import uniform_sampling
from blip_captioner import load_blip_model, frames_to_caption

# Load BLIP
load_blip_model()

Loading BLIP model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

BLIP model loaded on cuda!


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

(BlipProcessor:
 - image_processor: BlipImageProcessorFast {
   "data_format": "channels_first",
   "do_convert_rgb": true,
   "do_normalize": true,
   "do_rescale": true,
   "do_resize": true,
   "image_mean": [
     0.48145466,
     0.4578275,
     0.40821073
   ],
   "image_processor_type": "BlipImageProcessorFast",
   "image_std": [
     0.26862954,
     0.26130258,
     0.27577711
   ],
   "resample": 3,
   "rescale_factor": 0.00392156862745098,
   "size": {
     "height": 384,
     "width": 384
   }
 }
 
 - tokenizer: BertTokenizer(name_or_path='Salesforce/blip-image-captioning-base', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
 	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	100: AddedToken("[UNK]", rstrip=False, lstrip=False, sin

Run for Strategy A on Full Dataset

In [8]:
import os

path = '/content/drive/MyDrive/thesis-data/captions/strategy_A_full.json'

if os.path.exists(path):
    os.remove(path)
    print("Deleted! Strategy A will start fresh.")
else:
    print("File doesn't exist anyway — already fresh!")

Deleted! Strategy A will start fresh.


In [9]:
video_dir = '/content/drive/MyDrive/thesis-data/videos/'
save_path = '/content/drive/MyDrive/thesis-data/captions/strategy_A_full.json'

# Load existing checkpoint if any
if os.path.exists(save_path):
    with open(save_path, 'r') as f:
        results = json.load(f)
    print(f"Resuming from checkpoint: {len(results)} videos already done")
else:
    results = {}
    print("Starting fresh")

# Get all video IDs to process
all_video_ids = [f'video{i}' for i in range(7010)]
remaining = [v for v in all_video_ids if v not in results]
print(f"Remaining to process: {len(remaining)}")

# Run with checkpoint saving every 50 videos
start_time = time.time()
errors = []

for i, video_name in enumerate(remaining):
    video_path = os.path.join(video_dir, f'{video_name}.mp4')

    try:
        frames, fps, total = extract_frames(video_path)
        keyframes, indices = uniform_sampling(frames, K=8)
        caption = frames_to_caption(keyframes)

        results[video_name] = {
            'caption': caption,
            'keyframe_indices': indices,
            'total_frames': len(frames)
        }

    except Exception as e:
        errors.append(video_name)
        results[video_name] = {
            'caption': '',
            'keyframe_indices': [],
            'total_frames': 0,
            'error': str(e)
        }

    # Save checkpoint every 50 videos
    if (i+1) % 50 == 0:
        with open(save_path, 'w') as f:
            json.dump(results, f)
        elapsed = time.time() - start_time
        rate = (i+1) / elapsed
        remaining_time = len(remaining) - (i+1)
        eta = remaining_time / rate / 60
        print(f"  {len(results)}/7010 done | {elapsed/60:.1f} min elapsed | ETA: {eta:.1f} min")

# Final save
with open(save_path, 'w') as f:
    json.dump(results, f)

elapsed = time.time() - start_time
print(f"\nStrategy A complete!")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Errors: {len(errors)}")

# Final save of captions
with open(save_path, 'w') as f:
    json.dump(results, f)

elapsed = time.time() - start_time
print(f"\nStrategy A complete!")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Errors: {len(errors)}")

# Save timing to Drive immediately
timing_path = '/content/drive/MyDrive/thesis-data/results/timing_results.json'

if os.path.exists(timing_path):
    with open(timing_path, 'r') as f:
        timing_results = json.load(f)
else:
    timing_results = {}

timing_results['Strategy_A'] = {
    'total_time_minutes': round(elapsed / 60, 2),
    'total_videos': len(results),
    'avg_time_per_video_seconds': round(elapsed / len(results), 3)
}

with open(timing_path, 'w') as f:
    json.dump(timing_results, f, indent=2)

print(f"Avg per video: {elapsed / len(results):.3f} seconds")
print(f"Timing saved to Drive!")

Starting fresh
Remaining to process: 7010
  50/7010 done | 1.4 min elapsed | ETA: 200.9 min
  100/7010 done | 2.8 min elapsed | ETA: 192.0 min
  150/7010 done | 4.2 min elapsed | ETA: 191.0 min
  200/7010 done | 5.6 min elapsed | ETA: 189.8 min
  250/7010 done | 7.1 min elapsed | ETA: 192.7 min
  300/7010 done | 8.5 min elapsed | ETA: 189.2 min
  350/7010 done | 9.9 min elapsed | ETA: 189.0 min
  400/7010 done | 11.3 min elapsed | ETA: 187.5 min
  450/7010 done | 12.6 min elapsed | ETA: 184.0 min
  500/7010 done | 14.0 min elapsed | ETA: 182.9 min
  550/7010 done | 15.7 min elapsed | ETA: 184.2 min
  600/7010 done | 17.2 min elapsed | ETA: 184.0 min
  650/7010 done | 18.8 min elapsed | ETA: 183.5 min
  700/7010 done | 20.3 min elapsed | ETA: 183.1 min
  750/7010 done | 21.7 min elapsed | ETA: 181.1 min
  800/7010 done | 23.2 min elapsed | ETA: 180.1 min
  850/7010 done | 24.9 min elapsed | ETA: 180.7 min
  900/7010 done | 26.6 min elapsed | ETA: 180.3 min
  950/7010 done | 28.0 min ela

Run for Strategy B on Full Dataset

In [ ]:
from strategy_b_ssim import ssim_sampling

save_path_b = '/content/drive/MyDrive/thesis-data/captions/strategy_B_full.json'

if os.path.exists(save_path_b):
    with open(save_path_b, 'r') as f:
        results_b = json.load(f)
    print(f"Resuming from checkpoint: {len(results_b)} videos done")
else:
    results_b = {}

remaining_b = [f'video{i}' for i in range(7010) if f'video{i}' not in results_b]
print(f"Remaining: {len(remaining_b)}")

start_time = time.time()
errors_b = []

for i, video_name in enumerate(remaining_b):
    video_path = os.path.join(video_dir, f'{video_name}.mp4')

    try:
        frames, fps, total = extract_frames(video_path)
        keyframes, indices, _ = ssim_sampling(frames, threshold=0.7, max_keyframes=8)
        caption = frames_to_caption(keyframes)

        results_b[video_name] = {
            'caption': caption,
            'keyframe_indices': indices,
            'total_frames': len(frames)
        }

    except Exception as e:
        errors_b.append(video_name)
        results_b[video_name] = {
            'caption': '',
            'keyframe_indices': [],
            'total_frames': 0,
            'error': str(e)
        }

    if (i+1) % 50 == 0:
        with open(save_path_b, 'w') as f:
            json.dump(results_b, f)
        elapsed = time.time() - start_time
        rate = (i+1) / elapsed
        eta = (len(remaining_b) - (i+1)) / rate / 60
        print(f"  {len(results_b)}/7010 done | {elapsed/60:.1f} min elapsed | ETA: {eta:.1f} min")

with open(save_path_b, 'w') as f:
    json.dump(results_b, f)

elapsed = time.time() - start_time
print(f"\nStrategy B complete!")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Errors: {len(errors_b)}")


# Final save of captions
with open(save_path, 'w') as f:
    json.dump(results, f)

elapsed = time.time() - start_time
print(f"\nStrategy A complete!")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Errors: {len(errors)}")

# Save timing to Drive immediately
timing_path = '/content/drive/MyDrive/thesis-data/results/timing_results.json'

if os.path.exists(timing_path):
    with open(timing_path, 'r') as f:
        timing_results = json.load(f)
else:
    timing_results = {}

timing_results['Strategy_B'] = {
    'total_time_minutes': round(elapsed / 60, 2),
    'total_videos': len(results),
    'avg_time_per_video_seconds': round(elapsed / len(results), 3)
}

with open(timing_path, 'w') as f:
    json.dump(timing_results, f, indent=2)

print(f"Avg per video: {elapsed / len(results):.3f} seconds")
print(f"Timing saved to Drive!")

Run Strategy C on Full Dataset

In [ ]:
import json, time, os, sys
sys.path.append('/content/video-captioning-thesis/src')

from frame_extractor import extract_frames
from strategy_a_uniform import uniform_sampling
from blip_captioner import load_blip_model, frames_to_caption

from strategy_c_clip import load_clip_model, clip_kmeans_sampling

# Load both models
load_blip_model()
load_clip_model()

In [ ]:
save_path_c = '/content/drive/MyDrive/thesis-data/captions/strategy_C_full.json'

if os.path.exists(save_path_c):
    with open(save_path_c, 'r') as f:
        results_c = json.load(f)
    print(f"Resuming from checkpoint: {len(results_c)} videos done")
else:
    results_c = {}

remaining_c = [f'video{i}' for i in range(7010) if f'video{i}' not in results_c]
print(f"Remaining: {len(remaining_c)}")

start_time = time.time()
errors_c = []

for i, video_name in enumerate(remaining_c):
    video_path = os.path.join(video_dir, f'{video_name}.mp4')

    try:
        frames, fps, total = extract_frames(video_path)
        keyframes, indices = clip_kmeans_sampling(frames, K=8)
        caption = frames_to_caption(keyframes)

        results_c[video_name] = {
            'caption': caption,
            'keyframe_indices': indices,
            'total_frames': len(frames)
        }

    except Exception as e:
        errors_c.append(video_name)
        results_c[video_name] = {
            'caption': '',
            'keyframe_indices': [],
            'total_frames': 0,
            'error': str(e)
        }

    if (i+1) % 50 == 0:
        with open(save_path_c, 'w') as f:
            json.dump(results_c, f)
        elapsed = time.time() - start_time
        rate = (i+1) / elapsed
        eta = (len(remaining_c) - (i+1)) / rate / 60
        print(f"  {len(results_c)}/7010 done | {elapsed/60:.1f} min elapsed | ETA: {eta:.1f} min")

with open(save_path_c, 'w') as f:
    json.dump(results_c, f)

elapsed = time.time() - start_time
print(f"\nStrategy C complete!")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Errors: {len(errors_c)}")


# Final save of captions
with open(save_path, 'w') as f:
    json.dump(results, f)

elapsed = time.time() - start_time
print(f"\nStrategy A complete!")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"Errors: {len(errors)}")

# Save timing to Drive immediately
timing_path = '/content/drive/MyDrive/thesis-data/results/timing_results.json'

if os.path.exists(timing_path):
    with open(timing_path, 'r') as f:
        timing_results = json.load(f)
else:
    timing_results = {}

timing_results['Strategy_C'] = {
    'total_time_minutes': round(elapsed / 60, 2),
    'total_videos': len(results),
    'avg_time_per_video_seconds': round(elapsed / len(results), 3)
}

with open(timing_path, 'w') as f:
    json.dump(timing_results, f, indent=2)

print(f"Avg per video: {elapsed / len(results):.3f} seconds")
print(f"Timing saved to Drive!")